In [ ]:
Recommendation System
Number of community members for each anime.
Objective:
The objective of this assignment is to implement a recommendation system using cosine similarity on an anime dataset. 
Dataset:
Use the Anime Dataset which contains information about various anime, including their titles, genres,No.of episodes and user ratings etc.

Tasks:

Data Preprocessing:

Load the dataset into a suitable data structure (e.g., pandas DataFrame).
Handle missing values, if any.
Explore the dataset to understand its structure and attributes.

Feature Extraction:

Decide on the features that will be used for computing similarity (e.g., genres, user ratings).
Convert categorical features into numerical representations if necessary.
Normalize numerical features if required.

Recommendation System:

Design a function to recommend anime based on cosine similarity.
Given a target anime, recommend a list of similar anime based on cosine similarity scores.
Experiment with different threshold values for similarity scores to adjust the recommendation list size.
Analyze the performance of the recommendation system and identify areas of improvement.


In [ ]:
Unique ID of each anime.
Anime title.
Anime broadcast type, such as TV, OVA, etc.
anime genre.
The number of episodes of each anime.
The average rating for each anime compared to the number of users who gave ratings.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

# =============================
# Task 1: Data Preprocessing
# =============================

# Load dataset
file_path = "/Users/SS/Downloads/anime.csv"
df = pd.read_csv("/Users/SS/Downloads/anime.csv")
df

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266
...,...,...,...,...,...,...,...
12289,9316,Toushindai My Lover: Minami tai Mecha-Minami,Hentai,OVA,1,4.15,211
12290,5543,Under World,Hentai,OVA,1,4.28,183
12291,5621,Violence Gekiga David no Hoshi,Hentai,OVA,4,4.88,219
12292,6133,Violence Gekiga Shin David no Hoshi: Inma Dens...,Hentai,OVA,1,4.98,175


In [3]:
print("Shape:", df.shape)

Shape: (12294, 7)


In [4]:
print(df.head())

   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.16   

   members  
0   200630  
1   793665  
2   114262  
3   673572  
4   151266  


In [5]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB
None


In [6]:
# Handle missing values
# Fill rating with mean, drop rows with no name
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')

df['rating'] = df['rating'].fillna(df['rating'].mean())
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

df = df.dropna(subset=['name'])

# =============================
# Task 2: Feature Extraction
# =============================


# Convert genre text into dummy variables
df['genre'] = df['genre'].fillna('Unknown')
genre_dummies = df['genre'].str.get_dummies(sep=',')

# Select numerical features
num_features = df[['rating', 'episodes', 'members']].fillna(0)

# Scale numerical features
scaler = StandardScaler()
num_scaled = scaler.fit_transform(num_features)

# Combine features
features = np.hstack([genre_dummies.values, num_scaled])

# =============================
# Task 3: Cosine Similarity
# =============================

similarity_matrix = cosine_similarity(features)

# Create index mapping
anime_index = pd.Series(df.index, index=df['name']).drop_duplicates()

# =============================
# Task 4: Recommendation Function
# =============================

def recommend_anime(anime_name, top_n=5, threshold=0.5):
    if anime_name not in anime_index:
        return "Anime not found"
    
    idx = anime_index[anime_name]
    sim_scores = list(enumerate(similarity_matrix[idx]))
    
    # Sort by similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Filter by threshold and exclude itself
    sim_scores = [x for x in sim_scores if x[1] >= threshold and x[0] != idx]
    
    # Get top N
    sim_scores = sim_scores[:top_n]
    
    anime_indices = [i[0] for i in sim_scores]
    
    return df['name'].iloc[anime_indices]




print("\nRecommendations for Naruto:")
print(recommend_anime('Naruto', top_n=5, threshold=0.5))



print("\nMore recommendations (lower threshold):")
print(recommend_anime('Naruto', top_n=10, threshold=0.3))



print("\nInsights:")
print("- Genre plays a major role in similarity.")
print("- Ratings and members improve recommendation quality.")
print("- Lower threshold gives more results but less accuracy.")



Recommendations for Naruto:
288                Fairy Tail
582                    Bleach
6      Hunter x Hunter (2011)
304                D.Gray-man
440                Soul Eater
Name: name, dtype: object

More recommendations (lower threshold):
288                          Fairy Tail
582                              Bleach
6                Hunter x Hunter (2011)
304                          D.Gray-man
440                          Soul Eater
346                         Dragon Ball
1      Fullmetal Alchemist: Brotherhood
200                 Fullmetal Alchemist
86                   Shingeki no Kyojin
281                        Kill la Kill
Name: name, dtype: object

Insights:
- Genre plays a major role in similarity.
- Ratings and members improve recommendation quality.
- Lower threshold gives more results but less accuracy.
